# EDA-Q PDK Complete Workflow Demo (Based on README)

This Notebook corresponds to the complete hands-on workflow in `pdk/README.md`, including:

1. Converting process JSON to versioned PDK package
2. Loading PDK into `Design`
3. Explicitly reading generation default parameters and generating design objects
4. Auto-injecting routing default parameters and executing routing
5. Loading project overlay and verifying effects
6. Running `pdk doctor` and outputting JSON report

Recommended to use `pyoccenv` as the Notebook kernel.

## 0) Prerequisites

- Current working directory is within the repository (or its subdirectory)
- Source process file exists: `产线一期倒装芯片流程_按工序分组.json`
- Kernel environment: `pyoccenv`

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import os

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "pdk").exists() and (p / "api").exists():
            return p
    raise RuntimeError("Cannot locate repo root containing both 'pdk' and 'api'.")

REPO = find_repo_root()
os.chdir(REPO)

RAW_PROCESS_JSON = REPO / "产线一期倒装芯片流程_按工序分组.json"
PDK_TARGET = REPO / "pdk" / "foundries" / "sc_flipchip" / "line1" / "1.0.0"
DOCTOR_REPORT = REPO / "pdk" / "reports" / "doctor_report_demo.json"

print("REPO:", REPO)
print("RAW_PROCESS_JSON exists:", RAW_PROCESS_JSON.exists())
print("PDK_TARGET:", PDK_TARGET)


REPO: g:\EDA-Qv3
RAW_PROCESS_JSON exists: True
PDK_TARGET: g:\EDA-Qv3\pdk\foundries\sc_flipchip\line1\1.0.0


In [2]:
def run(cmd, cwd=REPO):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    res = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if res.stdout.strip():
        print(res.stdout)
    if res.returncode != 0:
        if res.stderr.strip():
            print(res.stderr)
        raise RuntimeError(f"Command failed with code {res.returncode}")
    return res


## 1) Convert Process JSON to Versioned PDK Package

In [3]:
run([
    sys.executable,
    str(REPO / "pdk" / "tools" / "convert_line1_process_json.py"),
    "--source", str(RAW_PROCESS_JSON),
    "--target", str(PDK_TARGET),
])

print("Generated package files:")
for p in sorted(PDK_TARGET.glob("*")):
    print(" -", p.name)


$ d:\ProgramData\anaconda3\envs\pyoccenv\python.exe g:\EDA-Qv3\pdk\tools\convert_line1_process_json.py --source g:\EDA-Qv3\产线一期倒装芯片流程_按工序分组.json --target g:\EDA-Qv3\pdk\foundries\sc_flipchip\line1\1.0.0
PDK package written to: g:\EDA-Qv3\pdk\foundries\sc_flipchip\line1\1.0.0

Generated package files:
 - design_rules.json
 - device_presets.json
 - layers.json
 - manifest.json
 - process_flow.json
 - routing_profile.json
 - source


## 2) Check PDK Core File Contents

In [4]:
manifest = json.loads((PDK_TARGET / "manifest.json").read_text(encoding="utf-8"))
layers = json.loads((PDK_TARGET / "layers.json").read_text(encoding="utf-8"))
design_rules = json.loads((PDK_TARGET / "design_rules.json").read_text(encoding="utf-8"))

summary = {
    "pdk_id": manifest["pdk_id"],
    "version": manifest["version"],
    "layer_count": len(layers.get("layers", [])),
    "hard_rules": design_rules.get("hard_rules", {}),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "pdk_id": "sc_flipchip_line1",
  "version": "1.0.0",
  "layer_count": 5,
  "hard_rules": {
    "min_linewidth_um": {
      "scope": "layer1",
      "value": 1.0,
      "source": "设计规则.DCR规则.最小线宽"
    },
    "min_spacing_um": {
      "scope": "layer1",
      "value": 1.0,
      "source": "设计规则.DCR规则.最小间距"
    }
  }
}


## 3) Load PDK and View Default Parameters

In [5]:
from api.design import Design

design = Design()
loaded_manifest = design.load_pdk("sc_flipchip_line1", version="1.0.0", profile="default")
print("Loaded:", loaded_manifest["pdk_id"], loaded_manifest["version"])

q_defaults = dict(design.get_pdk_generation_defaults("qubits"))
rd_defaults = dict(design.get_pdk_generation_defaults("readout_lines"))
chip_defaults = dict(design.get_pdk_generation_defaults("chips"))

print("qubits defaults:", q_defaults)
print("readout defaults:", rd_defaults)
print("chips defaults:", chip_defaults)


Loaded: sc_flipchip_line1 1.0.0
qubits defaults: {'qubits_type': 'Transmon', 'chip_name': 'chip0', 'dist': 2000}
readout defaults: {'rdls_type': 'ReadoutCavityFlipchip', 'chip_name': 'chip0'}
chips defaults: {'chip_name': 'chip0', 'chip_type': 'RecChip'}


## 4) Generate Basic Design (Explicitly Pass Generation Defaults)

Note: According to current architecture constraints, `generate_*` methods recommend explicit parameter passing, without auto-injection.

In [ ]:
design.generate_topology(topo_col=6, topo_row=6)
design.generate_qubits(topology=True, **q_defaults)
design.generate_readout_lines(qubits=True, **rd_defaults)

chip_name = q_defaults.get("chip_name", "chip0")
# design.add_chip(chip_name=chip_name, chip_type=chip_defaults.get("chip_type", "RecChip"))
design.generate_chip(qubits=True, dist = 4000)
design.gds.chips.copy_chip(old_chip_name="chip0", new_chip_name="chip1")

print("qubits count:", len(design.gds.qubits.options))
print("readout count:", len(design.gds.readout_lines.options))
print("chips:", list(design.gds.chips.options.keys()))

## 5) Routing (Auto-inject Routing Defaults) and Stage Validation

In [ ]:
routing_ops = dict(design._merge_pdk_routing_defaults({}))
print("Routing defaults preview:", routing_ops)

# Compatible with older PDK versions: if pins_geometric_ops is missing, add a minimal default for LaunchPad.
if ("pins_geometric_ops" not in routing_ops) or (not routing_ops.get("pins_geometric_ops")):
    routing_ops["pins_geometric_ops"] = {
        "trace_width": 15,
        "trace_gap": 5,
        "taper_height": 120,
        "pad_width": 120,
        "pad_height": 125,
        "pad_gap": 100,
        "orientation": 0,
        "start_straight": 50,
        "distance_to_chip": 350,
        "distance_to_qubits": 3650,
    }

# routing.Flipchip currently requires each readout line to have a numeric end_pos.
# If the current rdls_type is incompatible (e.g., ReadoutCavityFlipchip), fall back to ReadoutCavity for routing demo.
sample_rdl = next(iter(design.gds.readout_lines.options.values())) if len(design.gds.readout_lines.options) > 0 else None
need_fix = False
if sample_rdl is not None:
    end_pos = sample_rdl.get("end_pos", None)
    if (not isinstance(end_pos, (list, tuple))) or len(end_pos) < 2 or (not isinstance(end_pos[1], (int, float))):
        need_fix = True

if need_fix:
    print("Detected readout_lines incompatible with Flipchip routing. Regenerating with rdls_type='ReadoutCavity'.")
    design.generate_readout_lines(qubits=True, rdls_type="ReadoutCavity", chip_name=q_defaults.get("chip_name", "chip0"))

design.gds.routing(**routing_ops)

for stage in ["pre_layout", "pre_route", "pre_tapeout"]:
    report = design.validate_pdk(stage=stage)
    print(stage, "valid=", report["valid"], "errors=", len(report["errors"]), "warnings=", len(report["warnings"]))
design.gds.show_gds()

## 6) Apply Project Overlay

Example overlay: `pdk/projects/demo_flipchip_project/overlay.json`.

In [8]:
available_overlays = design.show_available_overlays()
print("available overlay ids:", list(available_overlays.keys()))

overlay_doc = design.load_project_overlay("demo_flipchip_project")
print("loaded overlay:", overlay_doc["overlay_id"])
print("active profile:", design.pdk_profile)
print("routing defaults after overlay:", dict(design._merge_pdk_routing_defaults({})))

if "chip1" not in design.gds.chips.options and "chip0" in design.gds.chips.options:
    design.copy_chip("chip0", "chip1")

overlay_report = design.validate_pdk(stage="pre_route")
print("overlay pre_route valid:", overlay_report["valid"], "errors:", len(overlay_report["errors"]), "warnings:", len(overlay_report["warnings"]))


['demo_flipchip_project']
available overlay ids: ['demo_flipchip_project']
loaded overlay: demo_flipchip_project
active profile: default
routing defaults after overlay: {'method': 'Flipchip_routing', 'chip_name': 'chip1', 'pins_type': 'LaunchPad', 'tmls_type': 'TransmissionPath', 'ctls_type': 'ChargeLine', 'pins_geometric_ops': {'trace_width': 15, 'trace_gap': 5, 'taper_height': 120, 'pad_width': 120, 'pad_height': 125, 'pad_gap': 100, 'orientation': 0, 'start_straight': 50, 'distance_to_chip': 350, 'distance_to_qubits': 3650}}
overlay pre_route valid: True errors: 0 warnings: 0


## 7) Export a Sample GDS

In [9]:
gds_path = REPO / "test" / "pdk_demo" / "demo_with_pdk.gds"
saved_path = design.gds.save_gds(path=str(gds_path))
print("GDS saved:", saved_path)


GDS saved: g:\EDA-Qv3\test\pdk_demo\demo_with_pdk.gds


## 8) Run pdk doctor and Output JSON Report

In [10]:
DOCTOR_REPORT.parent.mkdir(parents=True, exist_ok=True)
run([
    sys.executable,
    str(REPO / "pdk" / "tools" / "pdk_doctor.py"),
    "--pdk-id", "sc_flipchip_line1",
    "--version", "1.0.0",
    "--overlay-id", "demo_flipchip_project",
    "--json-out", str(DOCTOR_REPORT),
])

doctor_json = json.loads(DOCTOR_REPORT.read_text(encoding="utf-8"))
print(json.dumps(doctor_json, ensure_ascii=False, indent=2))


$ d:\ProgramData\anaconda3\envs\pyoccenv\python.exe g:\EDA-Qv3\pdk\tools\pdk_doctor.py --pdk-id sc_flipchip_line1 --version 1.0.0 --overlay-id demo_flipchip_project --json-out g:\EDA-Qv3\pdk\reports\doctor_report_demo.json
== PDK Doctor Summary ==
Total: 1, OK: 1, Failed: 0

[OK] sc_flipchip_line1@1.0.0
  - INFO: Overlay applied successfully.

JSON report written to: g:\EDA-Qv3\pdk\reports\doctor_report_demo.json

{
  "ok": 1,
  "failed": 0,
  "targets": [
    {
      "pdk_id": "sc_flipchip_line1",
      "version": "1.0.0",
      "status": "ok",
      "errors": [],
      "warnings": [],
      "infos": [
        "Overlay applied successfully."
      ]
    }
  ]
}


## 9) Common Issues

1. `ModuleNotFoundError: No module named 'pdk'`
   - Please ensure the current working directory is in the repository root or its subdirectory.

2. `validate_pdk` shows `errors`
   - Common causes are missing qubits/chips generation, or required objects for readout/routing not yet built.

3. Overlay application fails (base mismatch)
   - Check that `base_pdk.pdk_id/version` in the overlay matches the currently loaded PDK.